# 33 — Cross-Attention: ChemBERTa Tokens × ESM-2 Residues → pEC50

Fuses compound and protein representations via learned cross-attention:
- Query: per-token ChemBERTa embeddings (L_c × 768) — SMILES tokens
- Key/Value: per-residue ESM-2 embeddings (294 × 320) — PXR residues

The compound tokens learn to attend to relevant protein residues,
creating a protein-conditional compound representation. Only the
cross-attention layers and regression head are trained; encoders frozen.

Architecture: project → multi-head cross-attn → mean-pool → FFN → pEC50

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, EsmModel

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42; N_FOLDS = 5
torch.manual_seed(SEED)
DEVICE = 'cpu'
print(f'PyTorch {torch.__version__}  device={DEVICE}')

PyTorch 2.11.0+cpu  device=cpu


In [2]:
# ── 2. Encoders (frozen) ──────────────────────────────────────────────────────
print('Loading ChemBERTa-77M-MTR...')
CHEM_MODEL = 'deepchem/ChemBERTa-77M-MTR'  # hidden_size=384; cached locally
chem_tok   = AutoTokenizer.from_pretrained(CHEM_MODEL)
chem_enc   = AutoModel.from_pretrained(CHEM_MODEL).eval()
for p in chem_enc.parameters(): p.requires_grad_(False)

print('Loading ESM-2 (8M)...')
ESM_MODEL  = 'facebook/esm2_t6_8M_UR50D'
esm_tok    = AutoTokenizer.from_pretrained(ESM_MODEL)
esm_enc    = EsmModel.from_pretrained(ESM_MODEL).eval()
for p in esm_enc.parameters(): p.requires_grad_(False)

D_CHEM = chem_enc.config.hidden_size  # 384 for ChemBERTa-77M-MTR
D_PROT = 320
print(f'ChemBERTa-77M-MTR d_chem={D_CHEM}  ESM-2 d_prot={D_PROT}')

Loading ChemBERTa-77M-MTR...


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: deepchem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
regression.out_proj.bias   | UNEXPECTED | 
regression.out_proj.weight | UNEXPECTED | 
norm_std                   | UNEXPECTED | 
regression.dense.weight    | UNEXPECTED | 
norm_mean                  | UNEXPECTED | 
regression.dense.bias      | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading ESM-2 (8M)...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ChemBERTa-77M-MTR d_chem=384  ESM-2 d_prot=320


In [3]:
# ── 3. Pre-compute PXR residue embeddings (one forward pass) ──────────────────
PXR_SEQ = ('GLTEEQRMMIRELMDAQMKTFDTTFSHFKNFRLPGVLSSGCELPESLQAPSREEAAKWSQVRKDLCS'
            'LKVSLQLRGEDGSVWNYKPPADSGGKEIFSLLPHMADMSTYMFKGIISFAKVISYFRDLPIEDQISL'
            'LKGAAFELCQLRFNTVFNAETGTWECGRLSYCLEDTAGGFQQLLLEPMLKFHYMLKKLQLHEEEYVL'
            'MQAISLFSPDRPGVLQHRVVDQLQEQFAITLKSYIECNRPQPAHRFLFLKIMAMLTELRSINAQHTQ'
            'RLLRIQDIHPFATPLMQELFGITGS')

with torch.no_grad():
    esm_in = esm_tok(PXR_SEQ, return_tensors='pt')
    esm_out = esm_enc(**esm_in)
    # (1, L+2, 320) → skip CLS(0) and EOS(-1)
    pxr_residues = esm_out.last_hidden_state[0, 1:-1, :]  # (L_p, 320)

print(f'PXR residue embeddings: {pxr_residues.shape}')

PXR residue embeddings: torch.Size([293, 320])


In [4]:
# ── 4. Pre-compute compound token embeddings (all train + test) ────────────────
CACHE_TR = DATA_PROCESSED / 'chemberta77_tokens_train.pt'
CACHE_TE = DATA_PROCESSED / 'chemberta77_tokens_test.pt'
MAX_LEN = 128; BATCH_SIZE = 32

def extract_token_embeddings(smiles_list, cache_path):
    if cache_path.exists():
        print(f'  Loading cached {cache_path.name}')
        return torch.load(str(cache_path), weights_only=False)
    all_embs, all_masks = [], []
    for i in range(0, len(smiles_list), BATCH_SIZE):
        batch = smiles_list[i:i+BATCH_SIZE]
        enc = chem_tok(batch, return_tensors='pt', padding='max_length',
                       truncation=True, max_length=MAX_LEN)
        with torch.no_grad():
            out = chem_enc(**enc)
        # (B, MAX_LEN, 768) — includes CLS, SEP, PAD
        all_embs.append(out.last_hidden_state.cpu())
        all_masks.append(enc['attention_mask'].cpu())
    embs  = torch.cat(all_embs,  dim=0)  # (N, 128, 768)
    masks = torch.cat(all_masks, dim=0)  # (N, 128)
    result = (embs, masks)
    torch.save(result, str(cache_path))
    return result

train = load_train()
te    = load_test()
print('Extracting train token embeddings...')
embs_tr, masks_tr = extract_token_embeddings(train['smiles'].tolist(), CACHE_TR)
print('Extracting test token embeddings...')
embs_te, masks_te = extract_token_embeddings(te['smiles'].tolist(), CACHE_TE)
print(f'Train: {embs_tr.shape}  Test: {embs_te.shape}')

Extracting train token embeddings...
  Loading cached chemberta77_tokens_train.pt


Extracting test token embeddings...
  Loading cached chemberta77_tokens_test.pt
Train: torch.Size([4139, 128, 384])  Test: torch.Size([513, 128, 384])


In [5]:
# ── 5. Cross-attention model ──────────────────────────────────────────────────
class CompProtCrossAttn(nn.Module):
    def __init__(self, d_c=384, d_p=320, d_model=128, n_heads=4, dropout=0.2):
        super().__init__()
        self.q_proj  = nn.Linear(d_c, d_model)
        self.kv_proj = nn.Linear(d_p, d_model)
        self.cross   = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(d_model)
        self.ff      = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, compound_tokens, attn_mask, protein_residues):
        # compound_tokens: (B, L_c, d_c)
        # attn_mask: (B, L_c) — 1=real, 0=pad
        # protein_residues: (L_p, d_p)
        B = compound_tokens.shape[0]
        q  = self.q_proj(compound_tokens)          # (B, L_c, d_model)
        kv = self.kv_proj(protein_residues)         # (L_p, d_model)
        kv = kv.unsqueeze(0).expand(B, -1, -1)     # (B, L_p, d_model)
        out, _ = self.cross(q, kv, kv)             # (B, L_c, d_model)
        out = self.norm(out)
        # Mean-pool over real tokens (exclude PAD)
        mask_f = attn_mask.float().unsqueeze(-1)    # (B, L_c, 1)
        pooled = (out * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        return self.ff(pooled).squeeze(-1)          # (B,)


class PXRDataset(Dataset):
    def __init__(self, embs, masks, labels):
        self.embs, self.masks, self.labels = embs, masks, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return self.embs[i], self.masks[i], torch.tensor(self.labels[i], dtype=torch.float32)

print('Model architecture defined.')

Model architecture defined.


In [6]:
# ── 6. Training helpers ───────────────────────────────────────────────────────
def train_one_fold(embs_tr, masks_tr, y_tr, embs_va, masks_va, y_va,
                   pxr_res, epochs=80, patience=12, lr=3e-4, bs=32):
    model = CompProtCrossAttn().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.MSELoss()

    ds_tr = PXRDataset(embs_tr, masks_tr, y_tr)
    dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True)

    best_val, best_state, wait = float('inf'), None, 0
    for ep in range(epochs):
        model.train()
        for emb_b, mask_b, y_b in dl_tr:
            emb_b, mask_b, y_b = emb_b.to(DEVICE), mask_b.to(DEVICE), y_b.to(DEVICE)
            opt.zero_grad()
            pred = model(emb_b, mask_b, pxr_res)
            loss = criterion(pred, y_b)
            loss.backward(); opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(embs_va.to(DEVICE), masks_va.to(DEVICE), pxr_res)
        val_loss = criterion(val_pred.cpu(), torch.tensor(y_va, dtype=torch.float32)).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = model(embs_va.to(DEVICE), masks_va.to(DEVICE), pxr_res).cpu().numpy()
    return val_pred, model

print('Training helpers defined.')

Training helpers defined.


In [7]:
# ── 7. Scaffold 5-fold CV ──────────────────────────────────────────────────────
y_tr      = train['pec50'].values
scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
pxr_res   = pxr_residues.to(DEVICE)

oof = np.full(len(y_tr), np.nan)
fold_raes = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    val_preds, _ = train_one_fold(
        embs_tr[tr_idx], masks_tr[tr_idx], y_tr[tr_idx],
        embs_tr[va_idx], masks_tr[va_idx], y_tr[va_idx],
        pxr_res
    )
    oof[va_idx] = val_preds
    fold_rae = rae_fn(y_tr[va_idx], val_preds)
    fold_raes.append(fold_rae)
    print(f'  Fold {fold_i+1}: RAE={fold_rae:.4f}')

oof_rae = rae_fn(y_tr, oof)
print(f'\nOOF RAE: {oof_rae:.4f}  (fold mean: {np.mean(fold_raes):.4f} ± {np.std(fold_raes):.4f})')
print(f'ChemBERTa-MTR LGBM (no attn): 0.5993')
print(f'Cross-attention (this):        {oof_rae:.4f}')

np.save(DATA_PROCESSED / 'oof_crossattn_chemberta_esm2.npy', oof)

  Fold 1: RAE=0.5465


  Fold 2: RAE=0.6095


  Fold 3: RAE=0.6601


  Fold 4: RAE=0.6249


  Fold 5: RAE=0.6591

OOF RAE: 0.6147  (fold mean: 0.6200 ± 0.0416)
ChemBERTa-MTR LGBM (no attn): 0.5993
Cross-attention (this):        0.6147


In [8]:
# ── 8. Full retrain + test predictions ────────────────────────────────────────
_, final_model = train_one_fold(
    embs_tr, masks_tr, y_tr,
    embs_tr[-200:], masks_tr[-200:], y_tr[-200:],  # small held-out for early stopping
    pxr_res, epochs=100, patience=15
)

final_model.eval()
all_te_preds = []
BATCH = 64
with torch.no_grad():
    for i in range(0, len(embs_te), BATCH):
        p = final_model(
            embs_te[i:i+BATCH].to(DEVICE),
            masks_te[i:i+BATCH].to(DEVICE),
            pxr_res
        ).cpu().numpy()
        all_te_preds.append(p)

te_preds = np.concatenate(all_te_preds)
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED / 'te_crossattn_chemberta_esm2.npy', te_preds)

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'SMILES': te['smiles'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '33_crossattn_chemberta_esm2.csv'
sub.to_csv(out, index=False)
print(f'Saved: {out}  |  OOF RAE: {oof_rae:.4f}')
print(f'Test preds: mean={te_preds.mean():.3f}  std={te_preds.std():.3f}')

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\33_crossattn_chemberta_esm2.csv  |  OOF RAE: 0.6147
Test preds: mean=4.577  std=0.586
